# Machine Learning-Based Fraud Detection System

## Project Goal
The goal of this project is to develop a machine learning–based fraud detection system that can accurately identify whether a financial transaction is legitimate or fraudulent. Using anonymized real-world credit card transaction data, the project aims to:
- Analyze patterns in large-scale financial transactions
- Handle extreme class imbalance (only ~0.17% fraud cases)
- Build and compare predictive models such as Logistic Regression, LDA, Decision Trees, and Random Forest
- Evaluate models using metrics beyond accuracy, such as precision, recall, and confusion matrix, to ensure low false-negative rates
- Support real-time fraud detection efforts in financial institutions by identifying suspicious activities early

## Data Acquisition & Exploration

### 1. Load and Interpret Dataset
We start by loading the dataset and performing an initial inspection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, accuracy_score

import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('Fraud.csv')

# Initial display
df.head()

### 2. Inspect and Summarize Dataset Structure
Inspecting shape, data types, and statistical descriptions.

In [ ]:
# Shape of the dataset
print(f"Dataset Shape: {df.shape}")

# Data types
print("\nData Types:")
print(df.dtypes)

# Statistical summary
print("\nStatistical Description:")
print(df.describe())

### 3. Identify Class Imbalance
Fraud detection datasets are typically highly imbalanced. We check the distribution of the target variable `isFraud`.

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_percent = df['isFraud'].value_counts(normalize=True) * 100

print("Class Distribution:")
print(fraud_counts)
print("\nPercentage Distribution:")
print(fraud_percent)

# Discussion on Imbalance:
# Extreme class imbalance means that a model predicting 'Normal' 
# for every transaction would achieve >99% accuracy but fail to detect any fraud. 
# This makes precision, recall, and F1-score more critical evaluation metrics than accuracy.

### 4. Verify Data Quality
Checking for missing values and nulls.

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

# Analysis:
# If there are no missing values, additional cleaning like imputation is not required. 
# However, we may need to handle categorical variables and drop non-predictive columns like IDs.

## Feature Engineering & Preprocessing

### 5. Separate Input Features and Target Variable
We drop non-essential columns (`nameOrig`, `nameDest`) and encode categorical features.

In [ ]:
# Encoding categorical 'type' column
le = LabelEncoder()
df['type'] = le.fit_transform(df['type'])

# Features (X) and Target (y)
X = df.drop(['isFraud', 'isFlaggedFraud', 'nameOrig', 'nameDest'], axis=1)
y = df['isFraud']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

### 6. Apply Standardization / Scaling
Standardizing numeric features to ensure they have the same scale.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features have been standardized.")

### 7. Split Dataset (Stratified Sampling)
Using stratified sampling to preserve the class proportions in both training and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, stratify=y, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

## Model Building & Comparison

### 9. Train Classification Models
Training Logistic Regression, LDA, Decision Tree, and Random Forest.

In [ ]:
# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# Linear Discriminant Analysis
lda_model = LinearDiscriminantAnalysis()
lda_model.fit(X_train, y_train)
y_pred_lda = lda_model.predict(X_test)

# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

# Random Forest
rf_model = RandomForestClassifier(n_estimators=10, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("Models trained successfully.")

### 10. Evaluate Models and Compare Results
We present a structured comparison of all models.

In [ ]:
results = []

def evaluate_model(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    
    results.append({
        'Model': model_name,
        'Accuracy (%)': acc * 100,
        'Precision (%)': prec * 100,
        'Recall (%)': rec * 100
    })

evaluate_model(y_test, y_pred_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_lda, "Linear Discriminant Analysis")
evaluate_model(y_test, y_pred_dt, "Decision Tree")
evaluate_model(y_test, y_pred_rf, "Random Forest")

summary_df = pd.DataFrame(results)
pd.options.display.float_format = '{:,.2f}'.format
print("\nModel Comparison Summary:")
print(summary_df.to_string(index=False))

## Discussion on Model Issues and Bias

### 1. Handling "Y-Biasness" (Class Imbalance)
The dataset is extremely imbalanced (~0.13% fraud). This creates a significant bias towards the majority class (legitimate transactions). 
- **The Risk**: Models may achieve 99.9% accuracy simply by predicting every transaction as non-fraudulent. In such cases, the model is useless because its **Recall** (ability to catch fraud) would be 0%.
- **Our Solution**: We used **Stratified Sampling** to ensure that both training and testing sets maintain the same proportion of fraud cases. Additionally, we prioritized **Recall** and **Precision** over simple Accuracy.

### 2. Potential Prediction Issues
- **Overfitting**: High-capacity models like Random Forest might overfit the training data. If the model shows 100% accuracy on training but lower on testing, it is overfitting.
- **Data Leakage**: Features like `oldbalanceOrg` and `newbalanceOrig` are highly correlated with fraud in this synthetic dataset. If the relationship is *too* perfect, the model might fail in the real world where data is messier.
- **False Positives**: While catching fraud is the priority (Recall), a high number of false positives (low Precision) would result in blocking many legitimate customers, which is a significant business cost.

## Conclusion and Model Persistence

### Saving the Model for the Live Demo
To use this model in the live demo (`app.py`) without retraining, we save the trained model and the scaler to a file.

In [ ]:
# We save the Random Forest model as it generally provides high precision and recall
model_data = {
    'model': rf_model,
    'scaler': scaler
}

with open('model_data.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("Model and Scaler saved to 'model_data.pkl' successfully!")